# Notebook 4 - Masselot-main age-specific impact functions for heat mortality

This copied March2026 variant keeps the same hazards, exposures, vulnerability inputs, and downstream modules, but uses Masselot as the deterministic age-differentiated IF source and writes Burke only as sensitivity/generalizability output.

In [ ]:
import os
os.environ["URBAN_HEAT_OUTPUT_VARIANT"] = "masselot_main_agnostic"
os.environ["IF_MAIN_FAMILY"] = "masselot_tail"


In [ ]:
# City selector - the ONLY per-city line in this agnostic notebook.
# Set CITY to rome / athens / lisbon / copenhagen (any configured city).
import os
os.environ.setdefault("CITY", "Rome")


In [ ]:
# Bootstrap repo paths and the city-specific YAML configuration.
from pathlib import Path
import os, sys

def _find_root():
    start = Path.cwd()
    for cand in [start, *start.parents]:
        if (cand/"cityheat").is_dir() and (cand/"configs").is_dir():
            return cand
    raise RuntimeError("Repo root not found.")
ROOT = _find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from cityheat.nbsetup_masselot_main import bootstrap
from cityheat.paths import make_P, ensure_out

# Resolve the city slug from notebook globals or environment.
SLUG = globals().get("SLUG", os.environ["CITY"]).lower()

C    = bootstrap(SLUG)      # reads configs/<slug>.yml and syncs that city only if wanted
CFG  = C["CFG"]; CITY = C["CITY"]
BASE = C["BASE"]; OUT = C["OUT"]; INT = C["INT"]

P    = make_P(BASE)         # read-only path helper
OUTP = ensure_out(OUT)      # write-safe path helper
print(f"→ City: {CITY}  |  BASE={BASE}  OUT={OUT}  INT={INT}")

In [ ]:
cfg  = C.get("cfg")     # dict loaded from YAML

## Inputs from previous notebooks

This notebook starts from the harmonised products created in `NB02` and `NB03`. It loads:

- the daily `T2M` hazard object and its event table;
- age-specific exposure objects for `2020`, `2030`, `2040`, and `2050`;
- the city mask and gridded age arrays needed to rebuild the direct 2030 exposure if that file is missing.

The aim of this first block is simply to recover clean, year-specific CLIMADA-ready inputs before any impact-function work begins.


In [ ]:
print((INT / f"age_on_ref_{SLUG}_2030.npz").exists())
print((INT / "city_mask.npz").exists())
print((INT / "exposure_manifest.json").exists())

In [ ]:
# Load age-specific exposures for each target year, rebuilding the direct 2030 file if needed.
from climada.entity.exposures import Exposures
import numpy as np
import json
from pathlib import Path

SCEN = "SSP2"
YEARS = [2020, 2030, 2040, 2050]

def ensure_direct_2030_exposure_exists(SLUG=SLUG, INT=INT, OUT=OUT):
    candidates = [
        OUT / f"exposure_with_vulnerability_{SLUG}_2030.h5",
        INT / f"exposure_with_vulnerability_{SLUG}_2030.h5",
    ]
    for p in candidates:
        if p.exists():
            print(f"→ Found direct 2030 exposure: {p}")
            return p

    base_p = INT / f"exposure_with_vulnerability_{SLUG}.h5"
    if not base_p.exists():
        base_p = OUT / f"exposure_with_vulnerability_{SLUG}.h5"
    if not base_p.exists():
        raise FileNotFoundError(f"Missing baseline exposure template: {base_p}")

    exp2030 = Exposures.from_hdf5(base_p)

    age_npz_candidates = [
        INT / f"age_on_ref_{SLUG}_2030.npz",
    ]

    manifest_p = INT / "exposure_manifest.json"
    if manifest_p.exists():
        with open(manifest_p, "r") as f:
            manifest = json.load(f)
        try:
            age_npz_candidates.append(Path(manifest["direct_worldpop"]["2030"]["age_npz"]))
        except Exception:
            pass

    age_npz_path = next((p for p in age_npz_candidates if p.exists()), None)
    if age_npz_path is None:
        raise FileNotFoundError(
            "Could not find 2030 age grids from NB2. "
            "Expected something like age_on_ref_<slug>_2030.npz in interim."
        )

    city_mask_p = INT / "city_mask.npz"
    if not city_mask_p.exists():
        raise FileNotFoundError(f"Missing city mask: {city_mask_p}")
    CITY_MASK = np.load(city_mask_p)["city_mask"].astype(bool)

    age_npz = np.load(age_npz_path)
    arr_map = {
        "<15":   age_npz["lt15"],
        "15-64": age_npz["a15_64"],
        "65+":   age_npz["g65"],
    }

    if "age_group" not in exp2030.gdf.columns:
        raise KeyError("Exposure gdf has no 'age_group' column.")

    # Update the baseline exposure template with the 2030 age grids cell by cell.
    for grp, arr in arr_map.items():
        vals = arr[CITY_MASK].ravel().astype(float)
        idx = exp2030.gdf.index[exp2030.gdf["age_group"] == grp]

        if len(idx) != len(vals):
            raise ValueError(
                f"Row mismatch for {grp}: exposure rows={len(idx)} vs grid cells={len(vals)}"
            )

        exp2030.gdf.loc[idx, "value"] = vals

    exp2030.ref_year = 2030

    out_2030 = OUT / f"exposure_with_vulnerability_{SLUG}_2030.h5"
    exp2030.write_hdf5(out_2030)
    print(f"→ Built missing direct 2030 exposure: {out_2030}")
    return out_2030


def load_exposure_by_year(scen_code=SCEN, years=YEARS, SLUG=SLUG):
    exps = {}

    p2020 = INT / f"exposure_with_vulnerability_{SLUG}.h5"
    if not p2020.exists():
        p2020 = OUT / f"exposure_with_vulnerability_{SLUG}.h5"
    if not p2020.exists():
        raise FileNotFoundError(f"Missing baseline exposure for 2020: {p2020}")

    p2030 = ensure_direct_2030_exposure_exists(SLUG=SLUG, INT=INT, OUT=OUT)

    for y in years:
        if y == 2020:
            p = p2020
            label = "BASE"
        elif y == 2030:
            p = p2030
            label = "DIRECT"
        else:
            p = OUT / f"exposure_with_vulnerability_{SLUG}_{scen_code}_{y}.h5"
            if not p.exists():
                p = INT / f"exposure_with_vulnerability_{SLUG}_{scen_code}_{y}.h5"
            if not p.exists():
                raise FileNotFoundError(f"Expected exposure file not found: {p}")
            label = scen_code

        print(f"→ Loaded exposure for {label} {y}: {p}")
        exps[y] = Exposures.from_hdf5(p)

    return exps

exposures_by_year = load_exposure_by_year()

In [ ]:
# Load the daily T2M hazard produced earlier in the pipeline.
import pandas as pd
from climada.hazard import Hazard

ext_cfg = cfg.get("extreme_hazard", {}) or {}
track_default = "extreme" if bool(ext_cfg.get("enabled", False)) and bool(ext_cfg.get("run_extreme_track", False)) else "standard"
haz_track = str(os.environ.get("HAZARD_TRACK", track_default)).strip().lower()
use_extreme_track = haz_track in {"extreme", "event", "track_b", "heatwave"}
track_suffix = "_extreme" if use_extreme_track else ""

haz_candidates = [
    INT / f"hazard_T2M_daily_{SLUG}{track_suffix}.h5",
    OUT / f"hazard_T2M_daily_{SLUG}{track_suffix}.h5",
]

haz_path = next((p for p in haz_candidates if p.exists()), None)

if haz_path is None:
    raise FileNotFoundError(
        f"Could not find hazard file for {SLUG}. Looked for: "
        + ", ".join(str(p) for p in haz_candidates)
    )

H = Hazard.from_hdf5(haz_path)
print(f"→ Loaded hazard ({haz_track}): {haz_path}")
print(f"  events: {H.intensity.shape[0]}")
print(f"  cells:  {H.intensity.shape[1]}")

In [ ]:
# Reattach daily dates and per-year frequencies from the NB03 event table.
def _days_in_year(y):
    return 366 if (y % 4 == 0 and (y % 100 != 0 or y % 400 == 0)) else 365

def reattach_daily_dates(INT, SLUG, H, track_suffix="", strict_track=False):
    csv = INT / f"hazard_T2M_daily_events_{SLUG}{track_suffix}.csv"
    if strict_track and not csv.exists():
        raise FileNotFoundError(f"Missing events CSV for requested track: {csv}")
    if not csv.exists():
        # fallback to standard events CSV when running legacy hazards
        csv = INT / f"hazard_T2M_daily_events_{SLUG}.csv"
    df = pd.read_csv(csv).sort_values("row").reset_index(drop=True)

    n = H.intensity.shape[0]
    assert len(df) == n, f"Events CSV rows ({len(df)}) != hazard rows ({n})."

    H.event_id   = df["event_id"].to_numpy(int)
    H.date       = pd.to_datetime(df["date"]).to_numpy("datetime64[ns]")
    H.event_name = np.array([f"T2M_{d.date()}" for d in pd.to_datetime(df["date"])], dtype=object)

    yrs = pd.to_datetime(df["date"]).dt.year.to_numpy(int)
    H.frequency      = np.array([1.0/_days_in_year(y) for y in yrs], dtype=float)
    H.frequency_unit = "1/year"
    return H

H = reattach_daily_dates(INT, SLUG, H, track_suffix=track_suffix, strict_track=use_extreme_track)
H.haz_type = "T2M"
H.units    = "degC exceedance above T*" if use_extreme_track else "degC (daily mean)"

# Track-B hazards store exceedance (T - T*). IF curves are parameterised on absolute T.
# Convert only positive event-day exceedances back to absolute T at impact-evaluation time.
import json
EXTREME_TSTAR_C = None
if use_extreme_track:
    meta_path = INT / f"hazard_extreme_meta_{SLUG}.json"
    if not meta_path.exists():
        meta_path = OUT / f"hazard_extreme_meta_{SLUG}.json"
    if meta_path.exists():
        with open(meta_path) as f:
            _meta = json.load(f)
        t_star = _meta.get("threshold_degC", np.nan)
        try:
            EXTREME_TSTAR_C = float(t_star)
            if not np.isfinite(EXTREME_TSTAR_C):
                EXTREME_TSTAR_C = None
        except (TypeError, ValueError):
            EXTREME_TSTAR_C = None
    if EXTREME_TSTAR_C is None:
        raise RuntimeError(
            f"Extreme track requested but threshold_degC not found in {meta_path}. "
            "Run NB03 (extreme) first."
        )
    print(f"Extreme-track IF input transform enabled (T*={EXTREME_TSTAR_C:.2f}°C)")

yrs = pd.to_datetime(H.date).year.to_numpy(int)
print(pd.Series(yrs).value_counts().sort_index())
print(pd.DataFrame({"y": yrs, "f": H.frequency}).groupby("y")["f"].sum())

assert np.allclose(pd.DataFrame({"y": yrs, "f": H.frequency}).groupby("y")["f"].sum().values, 1.0)

if not hasattr(H, "date") or len(getattr(H, "date", [])) != H.intensity.shape[0]:
    raise RuntimeError("Daily dates not present in hazard. Re-run NB3 saving steps.")

dates = pd.to_datetime(pd.Series(H.date.astype("datetime64[ns]")))
year_idx = dates.dt.year.to_numpy(int)

In [ ]:
# Compute city-mean diagnostics using the hazard mask / fraction arrays.
# NOTE: under Track-B (extreme), hazard intensity is exceedance above T* and can be zero on non-event days.
num = H.intensity.multiply(H.fraction).sum(axis=1)   # sum over valid cells
den = H.fraction.sum(axis=1)                         # number of valid cells
citymean_daily_track = (num / den).A1                # track-native metric

citymean_daily = citymean_daily_track.copy()
citymean_for_tref = citymean_daily_track.copy()
citymean_label = "City-mean T2M (°C)"

if use_extreme_track:
    std_daily_csv = INT / f"hazard_events_T2M_daily_{SLUG}.csv"
    if std_daily_csv.exists():
        _std = pd.read_csv(std_daily_csv, parse_dates=["date"]).sort_values("date")
        citymean_daily = _std["mean_intensity_degC"].to_numpy(float)
        dates = pd.to_datetime(_std["date"]).reset_index(drop=True)
        year_idx = dates.dt.year.to_numpy(int)
        citymean_label = "City-mean T2M (°C) [standard hazard]"
        print(
            "Extreme track detected: using standard-hazard daily T2M for this diagnostic plot; "
            "track-native values are exceedances and can be zero on non-event days."
        )
    else:
        citymean_label = "City-mean exceedance (°C above T*)"
        print(
            "Extreme track detected and standard daily CSV not found; plotting exceedance values "
            "(all-zero means no detected event days)."
        )
else:
    citymean_for_tref = citymean_daily

pd.DataFrame({
    "date": dates,
    "citymean_degC": citymean_daily,
    "track_native_degC": citymean_daily_track,
}).head(10)

## Diagnostic check on the daily hazard

Before calibrating the impact functions, we inspect the city-mean daily `T2M` series as a simple diagnostic. This is **not** the final impact calculation. It is just a quick way to verify that the repaired hazard has the expected seasonal structure and to see which temperature range the city actually reaches before moving to the full grid-based analysis.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
# daily city-mean T2M per year (just for intuition)
daily_df = pd.DataFrame({"date": dates, "citymean_degC": citymean_daily})
daily_df["year"] = daily_df["date"].dt.year

fig, ax = plt.subplots(figsize=(6,3))
for y, sub in daily_df.groupby("year"):
    ax.plot(sub["date"], sub["citymean_degC"], label=y, alpha=0.7)
ax.set_ylabel(citymean_label)
ax.set_title(f"{CITY}: daily mean T2M by year")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## Main IF family and sensitivity registry

This variant uses the Masselot city- and age-specific impact functions as the deterministic downstream IF family. Burke polynomial and power-law curves are written only as sensitivity/generalizability artifacts.

## Masselot-main deterministic impact functions

The helper below loads the Masselot constant-tail and log-linear-tail IF JSONs built from the Zenodo/Masselot reconstruction workflow, promotes the configured `IF_MAIN_FAMILY` to the canonical downstream IF slot inside this variant output namespace, writes Masselot diagnostics, and copies the exact original March2026 Burke IF JSONs into this variant namespace as sensitivity curves.

In [ ]:
from cityheat.nb04_masselot_main import run_nb04_masselot_main

nb04_outputs = run_nb04_masselot_main(
    root=ROOT,
    out_dir=OUT,
    int_dir=INT,
    slug=SLUG,
    city=CITY,
    hazard=H,
    exposures_by_year=exposures_by_year,
)
nb04_outputs


## NB04 outputs

The canonical downstream files in `INT` now describe the configured Masselot main family, with metadata recording the true source and extrapolation assumption. The Burke polynomial and power-law files remain available only as sensitivity families.

In [ ]:
import json

with open(nb04_outputs["manifest"], "r") as f:
    if_family_manifest = json.load(f)

if_family_manifest
